# 04 — Methodology walkthrough

Step-by-step from raw data → cost matrices → M cost → solve → evaluate. For
new collaborators learning the pipeline before modifying it.

Reading order: top to bottom. Each section corresponds to one component of
the FGW formulation in `docs/methods.md`.

$$
\pi^* = \arg\min_{\pi}
  (1-\alpha) \cdot \langle M, \pi\rangle
  + \alpha \cdot \sum_{i,j,k,l} (C_m[i,k] - C_h[j,l])^2 \, \pi[i,j] \, \pi[k,l]
  - \varepsilon \cdot H(\pi)
$$

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore', category=DeprecationWarning, module='homer\\..*')

import numpy as np
import pandas as pd
from homer.data import load_cached, get_anchor_index, NETWORKS, assign_networks
from homer.viz.notebook import plot_brain_3d, plot_pi_heatmap

ROOT = Path.cwd().parent
ANN  = ROOT / 'outputs' / 'anndata'

## 1. Load the per-species AnnData

Each AnnData stores the mean FC matrix and per-node metadata (region, xyz, anchor flag, network).

In [2]:
M, _ = load_cached('mouse', cache_dir=ANN)
H, _ = load_cached('human', cache_dir=ANN)

for label, ad in (('mouse', M), ('human', H)):
    print(f'{label}: {ad.uns["n_nodes"]} nodes × {ad.uns["n_subjects"]} subjects')
    print(f'  fc_mean: {ad.uns["fc_mean"].shape} ({ad.uns["fc_mean"].dtype})')
    print(f'  garin anchors: {int(ad.var["garin_anchor"].sum())}')

mouse: 1864 nodes × 105 subjects
  fc_mean: (1864, 1864) (float32)
  garin anchors: 42
human: 2094 nodes × 113 subjects
  fc_mean: (2094, 2094) (float32)
  garin anchors: 42


## 2. Inspect the anchor index

The 42 Garin anchors per species form 21 (pair_id, hemisphere) putative homologue pairs. We sort both species the same way so the i-th mouse anchor matches the i-th human anchor.

In [3]:
idx_m = get_anchor_index(M.var)
idx_h = get_anchor_index(H.var)

print(f'mouse anchors: {len(idx_m)}')
print(f'human anchors: {len(idx_h)}')
print(f'sorted keys match? {idx_m.keys == idx_h.keys}')

# First 5 anchor pairs
pd.DataFrame({
    'pair_id':       idx_m.pair_ids[:5],
    'hemi':          idx_m.hemispheres[:5],
    'mouse_pos':     idx_m.pos[:5],
    'mouse_region':  M.var.iloc[idx_m.pos[:5]]['region'].values,
    'human_pos':     idx_h.pos[:5],
    'human_region':  H.var.iloc[idx_h.pos[:5]]['region'].values,
})

mouse anchors: 42
human anchors: 42
sorted keys match? True


,pair_id,hemi,mouse_pos,mouse_region,human_pos,human_region
0,1,L,0,L_Medial prefrontal cortex (mPFC),0,L_Medial prefrontal cortex (mPFC)
1,1,R,1,R_Medial prefrontal cortex (mPFC),1,R_Medial prefrontal cortex (mPFC)
2,2,L,2,L_Motor and premotor,2,L_Motor and premotor
3,2,R,3,R_Motor and premotor,3,R_Motor and premotor
4,3,L,4,L_Somatosensory cortex,4,L_Somatosensory cortex


## 3. Build the within-species relational cost matrices C_m and C_h

FC values lie in [-1, 1]. The simplest cost is `1 - r`, yielding a (n, n) symmetric, zero-diagonal distance matrix in [0, 2]. We then normalise by the max off-diagonal value so it lives in [0, 1] for stable mixing with other modalities.

In [4]:
from homer.costs import correlation_distance, normalise_cost, sc_correlation_distance

Cm_FC = normalise_cost(correlation_distance(M.uns['fc_mean'].astype(np.float64)), scheme='max')
Ch_FC = normalise_cost(correlation_distance(H.uns['fc_mean'].astype(np.float64)), scheme='max')
print(f'Cm_FC: {Cm_FC.shape}  off-diag mean={Cm_FC[~np.eye(Cm_FC.shape[0], dtype=bool)].mean():.3f}')
print(f'Ch_FC: {Ch_FC.shape}  off-diag mean={Ch_FC[~np.eye(Ch_FC.shape[0], dtype=bool)].mean():.3f}')

Cm_FC: (1864, 1864)  off-diag mean=0.814
Ch_FC: (2094, 2094)  off-diag mean=0.721


## 4. Mix in structural-connectivity cost (the production winner)

We weight `0.7·FC + 0.3·SC` for the production model. SC matrices were precomputed in `pipeline/03_build_costs.py`.

In [5]:
costs = np.load(ANN / 'full_costs.npz')
Cm_SC = costs['Cm_SC'].astype(np.float64)
Ch_SC = costs['Ch_SC'].astype(np.float64)
Cm = 0.7 * Cm_FC + 0.3 * Cm_SC
Ch = 0.7 * Ch_FC + 0.3 * Ch_SC
print(f'production Cm: shape={Cm.shape}, off-diag mean={Cm[~np.eye(Cm.shape[0], dtype=bool)].mean():.3f}')

production Cm: shape=(1864, 1864), off-diag mean=0.760


## 5. Build the cross-species cost matrix M

M has three components in the production model:
1. **xyz** — per-species-normalised Euclidean distance between mouse and human node coordinates. Spatial prior.
2. **anchor supervision** — for each visible anchor mouse position `mp`, set `M[mp, :] = 1.0` (forbid all other columns) and `M[mp, hp_correct] = 0` (free for the correct human partner).
3. (optional gene / network mask / M_anchor — off in production)

This is implemented inside `MultimodalFGW._solve()`. Below is what the M matrix looks like.

In [6]:
from homer.models.supervised import _build_xyz_M, _apply_anchor_supervision

M_xyz = _build_xyz_M(M.var, H.var)
print(f'M_xyz: shape={M_xyz.shape}, range [{M_xyz.min():.3f}, {M_xyz.max():.3f}]')

# Apply 0.5 weight + anchor supervision (full visibility — all 42 anchors)
M_full = 0.5 * M_xyz.copy()
visible = sorted(int(p) for p in idx_m.pair_ids)   # all anchors visible
M_full = _apply_anchor_supervision(M_full, idx_m, idx_h, visible, lam=1.0)
print(f'M (xyz + anchors): off-diag mean = {M_full[~np.isclose(M_full, 0)].mean():.3f}')
print(f'  cells set to lam=1.0 (forbidden): {int((M_full == 1.0).sum())}')
print(f'  cells set to 0    (allowed/free): {int((M_full == 0).sum())} '
       f'(includes 42 anchor diag + many non-anchor low-cost cells)')

M_xyz: shape=(1864, 2094), range [0.003, 1.000]
M (xyz + anchors): off-diag mean = 0.263
  cells set to lam=1.0 (forbidden): 164430
  cells set to 0    (allowed/free): 42 (includes 42 anchor diag + many non-anchor low-cost cells)


## 6. The mouse marginal

Semirelaxed FGW uses a fixed mouse marginal `p[i] = 1/n_mouse` and a free human marginal.

In [7]:
n_m = Cm.shape[0]
p = np.full(n_m, 1.0 / n_m)
print(f'p: shape={p.shape}, sum={p.sum():.6f}, each entry = {p[0]:.6e}')

p: shape=(1864,), sum=1.000000, each entry = 5.364807e-04


## 7. Call the solver

POT's `entropic_semirelaxed_fused_gromov_wasserstein` does the heavy lifting. With α=0.5 (equal FGW + W weight), ε=5e-3 (small entropic regularisation, hard solution), max_iter=25, tol=1e-5.

In [8]:
import ot, time
t0 = time.time()
pi, log = ot.gromov.entropic_semirelaxed_fused_gromov_wasserstein(
    M=M_full, C1=Cm, C2=Ch, p=p,
    alpha=0.5, epsilon=5e-3,
    max_iter=25, tol=1e-5, log=True,
)
print(f'solved in {time.time()-t0:.1f}s')
print(f'π shape: {pi.shape}, sum: {pi.sum():.3f} (mouse marginal = 1.0)')
print(f'srfgw_dist (loss): {log["srfgw_dist"]:.5f}')
print(f'mean row-max concentration: {(pi.max(axis=1) * n_m).mean():.3f}')

solved in 50.8s
π shape: (1864, 2094), sum: 1.000 (mouse marginal = 1.0)
srfgw_dist (loss): 0.01562
mean row-max concentration: 0.976


## 8. Same thing via the high-level model API

Identical solution, much less boilerplate. Use this for actual work.

In [9]:
from homer.models import MultimodalFGW
model = MultimodalFGW(use_sc=True, sc_weight=0.3, fc_weight=0.7,
                       epsilon=5e-3, xyz_weight=0.5)
model.fit(M, H, Cm_SC=Cm_SC, Ch_SC=Ch_SC)

# Same π?
print(f'max |pi_class - pi_manual| = {np.abs(pi - model.pi).max():.6f}')

max |pi_class - pi_manual| = 0.000000


## 9. The 42×42 anchor sub-block

If anchor supervision is working, this should be perfectly diagonal (each mouse anchor's row is one-hot at its known human partner).

In [10]:
pi_anchor = pi[np.ix_(idx_m.pos, idx_h.pos)]
diag_mass = float(np.diag(pi_anchor).sum() / pi_anchor.sum())
print(f'fraction of anchor sub-block mass on the diagonal: {diag_mass:.0%}')
plot_pi_heatmap(pi_anchor, title='Anchor sub-block (42×42)', width=500, height=500)

fraction of anchor sub-block mass on the diagonal: 100%


## 10. Held-out anchor CV — the real test

If we *withhold* the visual network's anchors from supervision, can the model still recover them via FC + SC + xyz alone?

In [11]:
from homer.data.anchors import held_out_metrics_graded

# Re-fit with visual anchors withheld
m_held = MultimodalFGW(use_sc=True, sc_weight=0.3, fc_weight=0.7,
                        epsilon=5e-3, xyz_weight=0.5)
m_held.fit(M, H, Cm_SC=Cm_SC, Ch_SC=Ch_SC, holdout_pair_ids=[5, 6])

pi_h = m_held.pi[np.ix_(idx_m.pos, idx_h.pos)]
metrics = held_out_metrics_graded(pi_h, idx_m, idx_h,
                                    held_out_pair_ids=[5, 6], var_h=H.var)
print('Visual held-out (pair_ids 5+6 = V1, V2):')
print(f'  top1 = {metrics["top1"]:.0%}  (visual is hard)')
print(f'  top5 = {metrics["top5"]:.0%}')
print(f'  pair_id = {metrics["pair_id"]:.0%}')
print(f'  mean_rank = {metrics["mean_rank"]:.1f}  (out of {metrics["max_rank_possible"]})')
print(f'  mean_xyz_dist = {metrics["mean_xyz_dist"]:.3f}  (lower = closer to truth)')

Visual held-out (pair_ids 5+6 = V1, V2):
  top1 = 50%  (visual is hard)
  top5 = 100%
  pair_id = 50%
  mean_rank = 1.5  (out of 4)
  mean_xyz_dist = 0.006  (lower = closer to truth)


## 11. FC translation Pearson r — the anchor-independent metric

Push mouse FC through π and compare predicted vs actual human FC. r ≈ 0.36 in production (vs 0.0 for random π).

In [12]:
from homer.eval import fc_translation_quality
net_h = assign_networks(H.var, idx_h)
res = fc_translation_quality(
    pi.astype(np.float64),
    M.uns['fc_mean'].astype(np.float64),
    H.uns['fc_mean'].astype(np.float64),
    network_labels_h=net_h,
)
for k, v in res.items():
    print(f'  {k:30s} {v}')

  pearson_r_overall              0.3610167715577442
  n_pairs_used                   950131
  n_human_nodes_kept             1379
  pred_mean                      0.02664097025990486
  pred_std                       0.06484740227460861
  actual_mean                    0.008539699684536814
  actual_std                     0.09250718744611418
  pearson_r_within_net           0.44350561115199244
  n_within_net                   119916
  pearson_r_cross_net            0.198247639314834
  n_cross_net                    830215
